In [1]:
import pandas as pd
import altair as alt

url = "https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/licenses_fall2022.csv"
df = pd.read_csv(url)

df.head()

,_id,License Type,Description,License Number,License Status,Business,Title,First Name,Middle,Last Name,...,Specialty/Qualifier,Controlled Substance Schedule,Delegated Controlled Substance Schedule,Ever Disciplined,LastModifiedDate,Case Number,Action,Discipline Start Date,Discipline End Date,Discipline Reason
0,1189509,DETECTIVE BOARD,PERMANENT EMPLOYEE REGISTRATION,129446286,NOT RENEWED,N,NaN,EILEEN,NaN,SANTACRUZ,...,NaN,NaN,NaN,N,03/18/2022,NaN,NaN,NaN,NaN,NaN
1,801037,DETECTIVE BOARD,FIREARM CONTROL CARD,229030294.0,NOT RENEWED,N,NaN,DAGMAR,J,NORDLUND,...,NaN,NaN,NaN,N,08/16/2006,NaN,NaN,NaN,NaN,NaN
2,365129,COSMO,LICENSED COSMETOLOGIST,11053076.0,NOT RENEWED,N,NaN,RADOJE,NaN,ZELENOVIC,...,NaN,NaN,NaN,N,05/26/2006,NaN,NaN,NaN,NaN,NaN
3,595427,COSMO,LICENSED COSMETOLOGIST,11295645.0,ACTIVE,N,NaN,BECKY SUE,L,BURROUGHS,...,NaN,NaN,NaN,N,11/12/2021,NaN,NaN,NaN,NaN,NaN
4,653668,COSMO,LICENSED NAIL TECHNICIAN,169006247,NOT RENEWED,N,NaN,BILL G,L,LETNER,...,NaN,NaN,NaN,N,05/30/2006,NaN,NaN,NaN,NaN,NaN


In [2]:
df.columns

Index(['_id', 'License Type', 'Description', 'License Number',
       'License Status', 'Business', 'Title', 'First Name', 'Middle',
       'Last Name', 'Prefix', 'Suffix', 'Business Name', 'BusinessDBA',
       'Original Issue Date', 'Effective Date', 'Expiration Date', 'City',
       'State', 'Zip', 'County', 'Specialty/Qualifier',
       'Controlled Substance Schedule',
       'Delegated Controlled Substance Schedule', 'Ever Disciplined',
       'LastModifiedDate', 'Case Number', 'Action', 'Discipline Start Date',
       'Discipline End Date', 'Discipline Reason'],
      dtype='object')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 31 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   _id                                      10000 non-null  int64  
 1   License Type                             10000 non-null  object 
 2   Description                              10000 non-null  object 
 3   License Number                           9940 non-null   object 
 4   License Status                           10000 non-null  object 
 5   Business                                 10000 non-null  object 
 6   Title                                    110 non-null    object 
 7   First Name                               9605 non-null   object 
 8   Middle                                   3622 non-null   object 
 9   Last Name                                9605 non-null   object 
 10  Prefix                                   3 non-

In [4]:
df['Expiration Date'].head()

0    09/30/2021
1    12/31/2003
2    09/30/1983
3    09/30/2023
4    10/31/2002
Name: Expiration Date, dtype: object

In [5]:
df['Expiration Date'] = pd.to_datetime(df['Expiration Date'], errors='coerce')

df['year'] = df['Expiration Date'].dt.year
df = df.dropna(subset=['year', 'License Type'])

# Chart 1

In [6]:
license_counts = (
    df.groupby('License Type')
      .size()
      .reset_index(name='count')
      .sort_values('count', ascending=False)
)
top_license_counts = license_counts.head(20)

In [7]:
chart1 = alt.Chart(top_license_counts).mark_bar().encode(
    x=alt.X('License Type:N', sort='-y', title='License Type'),
    y=alt.Y('count:Q', title='Number of Licenses'),
    color=alt.Color('License Type:N', legend=None)
).properties(
    title='Top 20 License Types by Count'
)

chart1

D:\Anacoda\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

This chart shows the distribution of all kinds of license. I intend to discover the most common license type and have a general acknowledge on distribution on each license. I used a bar chart because it can effectively compare counts across different categories. X axis represent the quantity, and Y axis is the type name of license. With different color for each bar, it could be clear to distinguish each type. The chart is sorted in descending order to see the most frequent license.

# Chart 2


In [8]:
year_type_counts = (
    df.groupby(['year', 'License Type'])
      .size()
      .reset_index(name='count')
)

In [9]:
top_types = (
    df.groupby('License Type')
      .size()
      .sort_values(ascending=False)
      .head(10)
      .index
      .tolist()
)
year_type_counts_top = year_type_counts[year_type_counts['License Type'].isin(top_types)]

In [10]:
years = sorted(year_type_counts_top['year'].dropna().unique().tolist())

year_dropdown = alt.binding_select(options=years, name='Select Year: ')
year_select = alt.selection_point(fields=['year'], bind=year_dropdown, value=years[0])

In [11]:
chart2 = alt.Chart(year_type_counts_top).mark_bar().encode(
    x=alt.X('License Type:N', sort='-y', title='License Type'),
    y=alt.Y('count:Q', title='Number of Licenses'),
    color=alt.Color('License Type:N', legend=None)
).add_params(
    year_select
).transform_filter(
    year_select
).properties(
    title='Top License Types by Selected Year'
)

chart2

D:\Anacoda\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

To enhance interactivity, I implemented a drop-down menu using Altair’s option binding feature. Users can select a specific year, and the system dynamically filters the dataset and updates the chart accordingly. This allows users to focus on a single time period at a time, rather than feeling overwhelmed by viewing all years simultaneously. Additionally, it facilitates comparisons of changes in the rankings and distributions of license types across different years, thereby improving the clarity and exploratory value of the charts.

In [13]:
import altair as alt
chart1.save("plot1.html")
chart2.save("plot2.html")

D:\Anacoda\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
D:\Anacoda\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
